In [3]:

import re, unicodedata
GAP = r"[^、。,;；\r\n]{0,10}"
WORSEN = (r"(?:出現|増悪|悪化|増強|増加|↑|強まっ|強くな|ひどく|増え|進行"
          r"|拡大|広が|目立ってき|出てき|出た|新規)")
EDEMA = r"(?:(?<!リンパ)(?<!リンパ性)(?<!薬剤性)(?<!本態性)(?<!全周性)浮腫|むくみ)"
DYSP = r"(?:呼吸苦|呼吸困難|息切れ|息苦し[いさ]?|起座呼吸)"
NEGATION_REGEX = re.compile(r"(なし|無し|なく|無く|認めず|認めない|否定|ない|\(\s*-\s*\))")
CONDITION_REGEX = re.compile(r"(時|場合|あれば|ようなら|際|連絡|ください|下さい|有無|確認|チェック|リスク|注意|既往|疑い|可能性|恐れ|おそれ)")
REASON_PATTERNS = {
 "weight_gain": [rf"体重{GAP}(?:増加|増|上昇)", r"WEIGHT GAIN", rf"(?<![A-Z])BW{GAP}(?:増加|増|UP|↑)"],
 "edema": [rf"{EDEMA}{GAP}{WORSEN}", rf"(?:新た|新規)に?{GAP}{EDEMA}",
           r"(?:WORSENING|NEW)\s+EDEMA", r"EDEMA.{0,10}(?:WORSEN|PROGRESS|INCREAS)"],
 "dyspnea": [rf"{DYSP}{GAP}{WORSEN}", rf"(?<![A-Z])DOE(?![A-Z]){GAP}{WORSEN}",
             rf"(?:新たに|新規){GAP}{DYSP}", r"(?:WORSENING|NEW\s+ONSET|NEW)\s+(?:DYSPNEA|ORTHOPNEA)"],
}
def has_pos(text, rx, w=10):
    for m in rx.finditer(text):
        tail = re.split(r"[、。\r\n]", text[m.end():m.end()+w])[0]
        if NEGATION_REGEX.search(m.group(0)) or NEGATION_REGEX.search(tail) or CONDITION_REGEX.search(tail): continue
        return True
    return False
def judge(reason, s):
    t = unicodedata.normalize("NFKC", s).upper()
    return int(has_pos(t, re.compile("|".join(REASON_PATTERNS[reason]), re.I)))
if __name__ == "__main__":
    T = [("edema","下腿浮腫が増悪",1),("edema","リンパ浮腫増悪",0),("edema","左下肢リンパ性浮腫が悪化",0),
     ("edema","薬剤性浮腫出現",0),("edema","本態性浮腫増悪",0),("edema","直腸全周性浮腫増悪",0),
     ("edema","下腿浮腫著明",0),("edema","浮腫:軽度",0),("edema","むくみ悪化",1),("edema","両足背に新たに浮腫",1),
     ("edema","浮腫増悪の有無",0),("edema","浮腫増悪なし、体重増加",0),("edema","浮腫なし、呼吸苦出現",0),
     ("dyspnea","呼吸苦出現、浮腫なし",1),("dyspnea","DOE増悪",1),("dyspnea","呼吸苦増悪時は連絡",0),("dyspnea","息切れ著明",0),
     ("weight_gain","体重2.5kg増加",1),("weight_gain","今日BW2kg増加",1),("weight_gain","体重増加なし",0)]
    ng=0
    for r,s,e in T:
        j=judge(r,s); ng+=(j!=e); print("OK " if j==e else "NG ", r, s, j)
    print("NG:",ng)


OK  edema 下腿浮腫が増悪 1
OK  edema リンパ浮腫増悪 0
OK  edema 左下肢リンパ性浮腫が悪化 0
OK  edema 薬剤性浮腫出現 0
OK  edema 本態性浮腫増悪 0
OK  edema 直腸全周性浮腫増悪 0
OK  edema 下腿浮腫著明 0
OK  edema 浮腫:軽度 0
OK  edema むくみ悪化 1
OK  edema 両足背に新たに浮腫 1
OK  edema 浮腫増悪の有無 0
OK  edema 浮腫増悪なし、体重増加 0
OK  edema 浮腫なし、呼吸苦出現 0
OK  dyspnea 呼吸苦出現、浮腫なし 1
OK  dyspnea DOE増悪 1
OK  dyspnea 呼吸苦増悪時は連絡 0
OK  dyspnea 息切れ著明 0
OK  weight_gain 体重2.5kg増加 1
OK  weight_gain 今日BW2kg増加 1
OK  weight_gain 体重増加なし 0
NG: 0
